# Day 4: A return as XML, read by tested Python

**Where in a return's file is the wages figure, and how do you write down where it is so that code finds it every time?**

Behind each case on the team's case list is a return, and the course's returns are XML files built to the IRS's Modernized e-File schemas, every value in them made up. Section 4.1 reads one by hand: its parts, a field's path through its namespace, the elements that may be missing or repeat, and the sorting of every field before any of it reaches code or a test.

From days 1 to 3, used as defined there: the team repository, a pull request and its review, a generated change, the coding agent, and tests that can fail.

The full reading for the day, with its sources: [the course web page](https://talent.catalyte.ai/courses/irs-bpa-to6/week-02/#day-4)

## 4.1 A return as an XML tree

*Where does the part about who files end, and what they reported begin?*

### A return is one XML file: a header, then the return data.

*The chapter's invented sample, every value made up.*

<img src="figures/a-return-is-one-xml-tree.png" width="864" alt="The chapter&#x27;s invented sample return, every value made up, drawn as boxes inside boxes. The outer box, Return: the root; its xmlns, http://www.irs.gov/efile, names every element&#x27;s namespace. Inside it, left, in berry: ReturnHeader, who files, holding PrimarySSN, a made-up testing number. Right, in blue: ReturnData, what they reported: the forms, holding, in orange, IRS1040: Form 1040; an attribute, documentId=&#x27;DOC0001&#x27;. Inside Form 1040: WagesAmt, its text: 41000; and DependentDetail, twice, drawn as a stack: each holds a DependentFirstNm.">

The claim in full: a return is one XML file: a root, Return, holding a header that identifies who files and the return data, the forms, Form 1040 among them.

Modernized e-File, the IRS system that receives returns electronically, defines them in XML: "A schema is an XML document that specifies the data elements, structure and rules for each form, schedule, document and/or attachment." For an individual return, "Return1040.xsd defines the XML structure for the return and wraps around the ReturnHeader1040 and ReturnData1040 XML documents in a transmission file." That is the IRS's Publication 4164, its guide to Modernized e-File.

On the left, ReturnHeader, who files, holding PrimarySSN, a made-up testing number. On the right, ReturnData, what they reported, holding the forms; Form 1040, IRS1040, with its attribute documentId='DOC0001', holds WagesAmt, whose text is 41000, and two DependentDetail elements, each holding a DependentFirstNm.

Python reads it as the format is built: "XML is an inherently hierarchical data format, and the most natural way to represent it is with a tree." Each node is an element with a tag, attributes, text and children (Python's documentation, xml.etree.ElementTree). On the course's sample, mark where the header ends and where Form 1040 begins.

### A name includes its namespace: search with a prefix map.

*Every unprefixed name sits in a namespace.*

- Search through NS, a dictionary of your own prefixes
- ElementTree reads only part of XPath: run every path

<img src="figures/the-full-name-carries-the-namespace.png" width="864" alt="Top: WagesAmt&#x27;s full name: a pair, written {http://www.irs.gov/efile}WagesAmt, the part in braces bracketed as the namespace&#x27;s URI, in blue, and WagesAmt as the local name. Two arrows down. Left, grey and dashed: By the local name: &#x27;WagesAmt&#x27;: find returns None. Right, orange: Through NS: &#x27;efile:WagesAmt&#x27;: found; NS maps efile to the URI.">

import xml.etree.ElementTree as ET NS = {'efile': 'http://www.irs.gov/efile'} root = ET.parse('sample_return.xml').getroot() wages = root.find('efile:ReturnData/efile:IRS1040/efile:WagesAmt', NS)

ElementTree reads only part of that path language, since "a full XPath engine is outside the scope of the module", so run every path you write down.

*Source: the chapter, section 6.2; the W3C, Namespaces in XML; Python's documentation, xml.etree.ElementTree; the IRS, MeF XML error troubleshooting*

### The schema says what may be missing or repeat, not the paper.

*Read each declaration in the schema:*

- minOccurs 1 or more: required
- maxOccurs: a number, or unbounded
- Both default to 1
- Absent is not a wage of zero

<img src="figures/four-rows-on-paper-100-in-the-schema.png" width="864" alt="Left, Paper Form 1040: four dependent rows, the first two holding the chapter&#x27;s made-up names TEST CHILD ONE and TEST CHILD TWO, two empty; 4 dependent rows. Right, in orange, In the MeF schema: a grid of 100 squares, ten by ten, one per entry, the first two filled for the same two dependents; 100 entries allowed; the sample fills 2.">

The next return has none, and another has no WagesAmt. You learn which fields behave like this from the schema, element by element, before your reader meets them.

The W3C's primer: "In general, an element is required to appear when the value of minOccurs is 1 or more." The most is maxOccurs, which "may be a positive integer such as 41, or the term unbounded to indicate there is no maximum number of occurrences." Both default to 1. The IRS's own example is your sample's: dependents, four rows on the paper Form 1040, while "the corresponding MeF schema allows 100 entries within this repeating group of elements." The figure draws the two, with the sample's two made-up dependents in each.

*Source: the chapter, section 6.3; the W3C, XML Schema Part 0: Primer; the IRS, Publication 4164*

### Sort every field: made-up, protected or instructor-only.

*Sort before any field reaches code, a fixture, a log or a prompt.*

| Label | What it means for a field | What you do |
| --- | --- | --- |
| Made-up test value | Invented for testing; identifies no one | Use it in fixtures: an IRS testing SSN has 00 as its 4th and 5th digits |
| Protected | In a real return, it would identify a person or hold their tax information | Never copy a real one, even with name and number deleted |
| Instructor-only | The course's own label: marked in its sample returns | Your trainer names these fields |

Because the reader will run on other returns in later weeks, and sorting is what keeps a real value out of its fixtures, logs and prompts.

The IRS's safeguards publication, Publication 1075, written for agencies that receive tax information, still lists as return information, which it protects, "Information collected by the IRS about any person’s tax affairs, even if identifiers, such as name, address and identification number are deleted", under the law it names: "Safeguarding FTI is critically important to continuously protect taxpayer confidentiality as required by IRC § 6103." FTI is Federal Tax Information; section 6103 of the Internal Revenue Code is the confidentiality law.

The IRS's test system makes a test value visible by rule, not by trust. Publication 1436: "Social Security Numbers valid for testing Federal and State returns must have “00” as the 4th and 5th digits."

**Check:** A search for WagesAmt finds nothing, though it is there. Why?

- **A.** The return has no wages
- **B.** Its full name includes the namespace's URI
- **C.** ElementTree cannot read IRS returns
- **D.** The path needs a position after each step

<details><summary>Answer</summary>

**B.** The root's xmlns puts every unprefixed name in its namespace: search with 'efile:WagesAmt' through NS.

</details>

**Discuss:** Why does a search for WagesAmt find nothing?

- When you open the file, where does the part about who files end, and the part about what they reported begin?
- Every value in the course's returns is made up. Why sort the fields at all?

Exercise slot: Write the path to each value in a sample return

## 4.2 A Python reader, tests first

*No WagesAmt, two dependents, a file cut off: what should the reader do?*

The question in full: your reader will meet a return with no WagesAmt, one with two dependents and one cut off halfway.

### Fixtures first: each one fixes what the reader must return.

*Each fixture, and what its test expects; every value made up.*

<img src="figures/one-fixture-per-case.png" width="864" alt="Four columns, every value made up. Each starts with a fixture file, in blue, and an arrow down to what its test expects. good.xml, two dependents: &#x27;41000&#x27;, and the two names; wages, and a list of dependents. no_dependents.xml, no DependentDetail: [ ], an empty list; dependents: none is a list too. no_wages.xml, no WagesAmt: in orange, None: absent; not a wage of zero. cut_off.xml, cut off halfway: ParseError, raised; it reaches the caller.">

Make one per case section 4.1 found: a good return with two dependents, one with none, one with no WagesAmt, one cut off halfway. pytest's parametrize runs one test over several inputs.

import pytest import xml.etree.ElementTree as ET from return_reader import read_values # not written yet

@pytest.mark.parametrize('name, expected', [ ('wages', '41000'), ('dependents', ['TEST CHILD ONE', 'TEST CHILD TWO']), ]) def test_good_return(name, expected): assert read_values('fixtures/good.xml')[name] == expected

Run the tests and watch each fail: the reader does not exist yet.

### Write the tests yourself; read with the parser in Python.

**The tests first, and yours**

- Percival: a comprehensive suite, run often, "a must-have"; and "too early to tell"
- Four teams at Microsoft and IBM: pre-release defect density down 40% to 90%
- 82 tasks, 39 professionals: small, even steps mattered; the order did not
- So the reason is another: one agent's reader and tests can share one mistake

**The parser already in Python**

- ElementTree: nothing to install, nothing to review
- ET.parse reads the file into a tree; getroot returns its root
- From the root, section 4.1's paths and NS reach each value
- Expat below 2.7.2 may be open to "billion laughs" and other attacks

Harry Percival, author of a book on test-driven development in Python: "When we’re thinking about the problem of hallucinations, it sure seems like having a comprehensive test suite and running it frequently, is going to be a must-have." He adds: "The truth is that it’s too early to tell."

In four teams at Microsoft and IBM, "the pre-release defect density of the four products decreased between 40% and 90% relative to similar projects that did not use the TDD practice." A study of 82 development tasks by 39 professionals found that small, even steps went with better results and the order did not: "Sequencing, the order in which test and production code are written, had no important influence." So the reason here is the chapter's section 4.2: a reader and tests written together by one coding agent can share one mistake.

The coding agent proposes a third-party package for the XML. ElementTree, the standard library's reader, is enough: ET.parse "Parses an XML section into an element tree", and getroot "Returns the root element for this tree." From the root, section 4.1's paths and namespace dictionary reach each named value.

### Test for absence with is None: a present wage tests False.

*ElementTree's lookups give the three shapes:*

- find: an element, or None
- findall: a list, in document order
- A cut-off file raises ParseError: let it reach the caller

<img src="figures/a-present-wage-tests-as-false.png" width="864" alt="Top: WagesAmt, present: its text 41000, a made-up value; no subelements. Two arrows down. Left, grey and dashed: if element: it tests as False: the wage is reported missing. Right, orange: if element is None: False, so the reader returns its text, 41000.">

find comes back empty-handed on the return with no WagesAmt. Something else: absent, because section 4.1 showed that absence means nothing in particular. ElementTree's lookups give the three shapes. find "Returns an element instance or None." findall "Returns a list containing all matching elements in document order." And findtext tells present-but-empty from absent: "Note that if the matching element has no text content an empty string is returned."

A file cut off halfway is an error, and the parser raises ParseError, the "XML parse error, raised by the various parsing methods in this module when parsing fails." Let it reach the caller: a reader that catches it and returns an empty result repeats the empty case list of the chapter's section 2.1.

*Source: the chapter, section 7.3; Python's documentation, xml.etree.ElementTree*

### Give the agent your tests, then name each error it makes.

*Given tests with the task, GPT-4 solved 12.0% and 8.5% more problems.*

| What the reader does | The study's name for it | What catches it |
| --- | --- | --- |
| Works only on the sample's shape, such as two dependents | Prompt-biased code | The no-dependents file |
| Calls getchildren(), deprecated in some Python versions | Close to Wrong Attribute | Any test, if Python lacks it |
| Passes the file's path to fromstring, which parses a string | Wrong Input Type | The good-return tests |
| Leaves the error branch as pass | Incomplete Generation | The cut-off file |

The coding agent's reader handles the good return, and your tests fail on the other three files. With GPT-4 on two public benchmarks of function-sized problems, "including tests contributes to solving an additional 12.0% and 8.5% of problems, respectively", against the same model given no tests.

Then name what comes back with the bug patterns of the study in the chapter's section 3.2, on bugs in code that language models generate. Prompt-biased code: it "excessively relies on provided examples or particular terms in the prompt"; the no-dependents file catches it. getchildren(), which the study saw models use on XML though it "is deprecated for certain versions of Python": close to Wrong Attribute, an "incorrect/non-existent attribute"; any test catches it, if the course's Python no longer has the method. fromstring "Parses an XML section from a string constant", so passing it the file's path is Wrong Input Type: it "uses an incorrect input type in a correct function call"; the good-return tests catch it. An error branch left as pass is Incomplete Generation; the cut-off file catches it.

The review names each error, its line, and the test that caught it.

**Check:** Your reader tests if element: on a present WagesAmt. Result?

- **A.** The wage, 41000
- **B.** The wage reported missing
- **C.** A ParseError
- **D.** An empty string

<details><summary>Answer</summary>

**B.** WagesAmt has no subelements, so it tests as False: test for absence with is None.

</details>

**Discuss:** Zero, an empty string, or something else?

- The agent could write the reader and its tests. Why write the tests first, yourself?
- find returns nothing for a missing WagesAmt. What should the reader report?
- The agent's reader fails three of your tests. What goes in the review?

Exercise slot: Read a return's values in Python, tests first

## The week's Labs

*Six Labs, one live build each*

### The week's six Labs: one live build each, in this order.

*Each Lab's build script, written by its author, is in today's notebook.*

| Lab | What you build |
| --- | --- |
| 1 | Add a feature to the Python service and web page with an approved coding tool |
| 2 | Review three generated pull requests; refuse one, with the evidence it lacked |
| 3 | An AI-written Playwright suite in the pipeline; a seeded regression blocked |
| 4 | The measured exercise: the same task with and without the coding agent |
| 5 | Annotate a sample return's XML tree by hand: its parts, paths to five values |
| 6 | Write tests and bad-input fixtures, a Python reader; review an agent's parser |

Labs 5 and 6 are today's two sections, applied: section 4.1 by hand, section 4.2 in tested Python.

## Recap

*What this week set up*

### What this week set up: three things later weeks use.

*Every later week works through reviewed pull requests and this pipeline.*

| What carries forward | Where it is used next |
| --- | --- |
| The pipeline that blocks a failing change | Week 5: a Playwright test for a model's score on screen; a model version promoted or rolled back through it |
| Refusing a generated change | Week 8: a legacy component a coding agent modernizes |
| The return reader | Week 3: returns parsed into keyed tables, a child table for each repeated element. Week 4: features from the Form 1040 body only |

The pipeline that blocks a failing change: week 5 adds a Playwright end-to-end test for the path to a model's score on screen, passing in this pipeline, and promotes or rolls back a model version through it on measured evidence, the chapter's section 5.5 step applied to a model.

Refusing a generated change: week 8 applies the reading of a generated change to a legacy component a coding agent modernizes, after characterization tests have pinned what it does today.

Week 4 builds features from the Form 1040 body only, never from a header field: the line you drew in section 4.1.

**Discuss:** What can you now show?

- Look back at what you said at the opening you wanted to show. What can you now show from the team repository alone?
- Which of the three things this week sets up will you reach for first?

## Close

*Key points, and what week 3 builds*

### Read the return by its schema; write the reader's tests first.

- A return is one XML file: a header, then the return data and its forms
- A name includes its namespace: search through NS, and run every path
- The schema says what may be missing or repeat; absent is not zero
- A real return with its name and number deleted is still protected
- Fixtures first; test absence with is None; let a ParseError reach the caller

### Absent is not zero.

- Week 3: your reader parses the returns
- Into keyed tables, a child table for each repeated element
- Week 4: features from the Form 1040 body only